# 🟢 Day 1 — Your First Graph

**LangGraph from Zero to Advanced · Day 1 of 16**

**Today's goal:** understand what LangGraph is and build your first graphs.

> No API key needed today. We learn the *graph machinery* with plain Python first. An LLM is just one kind of step inside a graph — we add it on Day 4.

| Lesson | Topic | Time |
|---|---|---|
| 1 | What is LangGraph? | 3 min |
| 2 | Setup | 2 min |
| 3 | Hello World graph | 10 min |
| 4 | See your graph (diagram) | 3 min |
| 5 | Two nodes in a row | 7 min |
| 6 | Nodes return only what changed | 5 min |
| 7 | Watch the graph step by step (`stream`) | 5 min |
| 8 | Practice | 10 min |

**How to use:** run each cell with `Shift + Enter`, top to bottom. After each lesson, change something and run it again.

## Lesson 1 — What is LangGraph?

LangGraph is a Python library for building programs as a **graph** — a flowchart that actually runs.

Think of a **food delivery order**:

```
 order placed ──► kitchen cooks ──► rider picks up ──► delivered
```

* 📋 **State** = the order slip. Every step can read it and write on it.
* 🟦 **Node** = one step (kitchen, rider). It's just a Python function.
* ➡️ **Edge** = the arrow that says which step comes next.
* 🚦 **START / END** = where the flow begins and stops.

| Idea | In code | Plain English |
|---|---|---|
| State | `class State(TypedDict)` | the shared data |
| Node | `def my_node(state): return {...}` | one unit of work |
| Edge | `builder.add_edge("a", "b")` | after `a`, run `b` |
| Compile | `app = builder.compile()` | turn the drawing into a runnable app |
| Run | `app.invoke({...})` | start with some input, get the final state |

**Why not just call functions one after another?** You can — until you need loops, branching, memory, human approval, or several AI agents working together. LangGraph gives you all of these. That's why it's the most popular way to build **AI agents** in Python.

## Lesson 2 — Setup

Run this once. (Better: from the repo root run `pip install -r requirements.txt` — it installs everything for all 16 days.)

In [ ]:
%pip install -q -U langgraph

In [ ]:
from importlib.metadata import version
print("LangGraph version:", version("langgraph"))

## Lesson 3 — Hello World graph

Every LangGraph program follows the same **5-step recipe**. Memorise it — you will use it every day.

```
1. Define the State        class State(TypedDict): ...
2. Write the nodes         def node(state): return {...}
3. Create a builder        builder = StateGraph(State); builder.add_node(...)
4. Connect with edges      builder.add_edge(START, "node") ...
5. Compile and run         app = builder.compile(); app.invoke({...})
```

We'll build the smallest possible graph:

```
START ──► greet ──► END
```

In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

# Step 1: the State — which keys exist and their types
class State(TypedDict):
    name: str
    greeting: str

In [ ]:
# Step 2: a node — a normal function. It READS the state and RETURNS an update.
def greet(state: State) -> dict:
    return {"greeting": f"Hello, {state['name']}! Welcome to LangGraph 👋"}

In [ ]:
# Step 3 + 4: builder, nodes and edges
builder = StateGraph(State)
builder.add_node("greet", greet)     # give the node a name + its function
builder.add_edge(START, "greet")     # first run "greet"
builder.add_edge("greet", END)       # then stop

In [ ]:
# Step 5: compile and run
app = builder.compile()
result = app.invoke({"name": "Asha"})
print(result)

### What just happened?

1. `invoke({"name": "Asha"})` created the starting state: `{"name": "Asha"}`.
2. LangGraph followed `START → greet` and called `greet(state)`.
3. `greet` returned `{"greeting": "..."}`. LangGraph **merged** it into the state.
4. LangGraph followed `greet → END` and returned the final state.

In [ ]:
# ✅ Check — every notebook has small checks like this. If it prints, you're good.
assert result["greeting"] == "Hello, Asha! Welcome to LangGraph 👋"
print("✅ Lesson 3 works!")

🧪 **Try it:** change `"Asha"` to your own name and run the cell again.

## Lesson 4 — See your graph

A compiled graph can draw itself. We write a tiny helper `show_graph()` that we'll reuse every day:

* it tries to draw a **picture** (needs internet), and
* if that fails, it prints the diagram as **text** — paste that into <https://mermaid.live> to see it.

In [ ]:
from IPython.display import Image, display

def show_graph(app):
    try:
        display(Image(app.get_graph().draw_mermaid_png()))
    except Exception:
        print(app.get_graph().draw_mermaid())

show_graph(app)

## Lesson 5 — Two nodes in a row

Real work has several steps. Each node can read what earlier nodes wrote.

```
START ──► clean ──► count_words ──► END
```

In [ ]:
class TextState(TypedDict):
    text: str
    clean_text: str
    word_count: int

def clean(state: TextState) -> dict:
    # remove extra spaces and make lower-case
    return {"clean_text": " ".join(state["text"].split()).lower()}

def count_words(state: TextState) -> dict:
    # this node uses what "clean" wrote
    return {"word_count": len(state["clean_text"].split())}

builder = StateGraph(TextState)
builder.add_node("clean", clean)
builder.add_node("count_words", count_words)
builder.add_edge(START, "clean")
builder.add_edge("clean", "count_words")
builder.add_edge("count_words", END)
text_app = builder.compile()

show_graph(text_app)

In [ ]:
out = text_app.invoke({"text": "   Hello     LangGraph   WORLD  "})
print(out)

In [ ]:
assert out["clean_text"] == "hello langgraph world"
assert out["word_count"] == 3
print("✅ Lesson 5 works!")

💡 **Shortcut:** for a straight line of nodes you can use `add_sequence`:

In [ ]:
builder = StateGraph(TextState)
builder.add_sequence([clean, count_words])   # node names = function names
builder.add_edge(START, "clean")
builder.add_edge("count_words", END)
print(builder.compile().invoke({"text": "one two three four"}))

## Lesson 6 — Nodes return only what changed

A node does **not** return the whole state. It returns a small dict with only the keys it wants to change. Other keys stay as they are.

Let's prove it: `step_b` returns nothing about `name`, yet `name` survives.

In [ ]:
class Profile(TypedDict):
    name: str
    age: int
    city: str

def step_a(state: Profile) -> dict:
    return {"age": 30}           # only age

def step_b(state: Profile) -> dict:
    return {"city": "Pune"}      # only city

b = StateGraph(Profile)
b.add_sequence([step_a, step_b])
b.add_edge(START, "step_a")
b.add_edge("step_b", END)

profile = b.compile().invoke({"name": "Ravi"})
print(profile)
assert profile == {"name": "Ravi", "age": 30, "city": "Pune"}
print("✅ Untouched keys are kept!")

⚠️ **Common mistake:** returning a key that is *not* in the State. LangGraph **silently ignores** it — no error, the value just disappears. Always add every key to the `TypedDict` first.

## Lesson 7 — Watch the graph step by step

`invoke()` gives you only the **final** state. `stream()` gives you a result **after every node** — perfect for learning and debugging.

| `stream_mode` | You get after each node |
|---|---|
| `"updates"` | only what that node returned: `{node_name: update}` |
| `"values"` | the full state |

In [ ]:
print("--- updates ---")
for chunk in text_app.stream({"text": " LangGraph  is  FUN "}, stream_mode="updates"):
    print(chunk)

print("\n--- values ---")
for state in text_app.stream({"text": " LangGraph  is  FUN "}, stream_mode="values"):
    print(state)

## Lesson 8 — Practice

**Exercise 1.** Build `START → shout → END`. State has `text` and `loud`. `shout` puts the upper-case text plus `"!!!"` into `loud`.

**Exercise 2.** Build a 3-node pipeline for a price: `add_tax` (18% GST) → `add_delivery` (+40) → `round_total` (round to 2 decimals). State: `price`, `total`.

Try first, then run the solutions below.

In [ ]:
# ✅ Solution 1
class ShoutState(TypedDict):
    text: str
    loud: str

def shout(state: ShoutState) -> dict:
    return {"loud": state["text"].upper() + "!!!"}

g = StateGraph(ShoutState)
g.add_node("shout", shout)
g.add_edge(START, "shout")
g.add_edge("shout", END)
res = g.compile().invoke({"text": "langgraph rocks"})
print(res)
assert res["loud"] == "LANGGRAPH ROCKS!!!"

In [ ]:
# ✅ Solution 2
class PriceState(TypedDict):
    price: float
    total: float

def add_tax(state: PriceState) -> dict:
    return {"total": state["price"] * 1.18}

def add_delivery(state: PriceState) -> dict:
    return {"total": state["total"] + 40}

def round_total(state: PriceState) -> dict:
    return {"total": round(state["total"], 2)}

g = StateGraph(PriceState)
g.add_sequence([add_tax, add_delivery, round_total])
g.add_edge(START, "add_tax")
g.add_edge("round_total", END)
res = g.compile().invoke({"price": 499})
print(res)
assert res["total"] == 628.82
print("✅ All exercises solved!")

## 🎯 Day 1 Recap

* A graph = **State** + **Nodes** + **Edges**, starting at `START` and finishing at `END`.
* A node is a plain function: `state in → small update dict out`.
* `compile()` makes it runnable; `invoke()` runs it; `stream()` shows every step.

**Tomorrow (Day 2):** what happens when two nodes write to the **same** key? Meet **reducers**.